In [1]:
import pandas as pd
from sqlalchemy import create_engine 
engine = create_engine('sqlite:///formula1.sqlite')

Ejercicio 1: INNER JOIN
Pregunta: ¿Cuáles son los nombres de las carreras y los nombres de los circuitos donde se llevaron a cabo esas carreras en la temporada 2023? Muestra solo las carreras que tienen un circuito asociado.

Concepto: Un INNER JOIN (o simplemente JOIN) devuelve solo las filas que tienen coincidencias en ambas tablas. Si una carrera no tiene un circuitId válido que coincida con un circuitId en la tabla circuits, esa carrera no aparecerá.

In [3]:
#Recordatorio: revisar teoria de conjuntos y mirarlo la proxima clase

query = """
            SELECT
                r.name AS race_name,
                c.name AS circuit_name,
                r.year
            FROM
                races AS r
            INNER JOIN
                circuits AS c 
            ON r.circuitId = c.circuitId
            WHERE
                r.year = 2023;
    """

df = pd.read_sql_query(query, engine)
df.head()

,race_name,circuit_name,year
0,Bahrain Grand Prix,Bahrain International Circuit,2023
1,Saudi Arabian Grand Prix,Jeddah Corniche Circuit,2023
2,Australian Grand Prix,Albert Park Grand Prix Circuit,2023
3,Azerbaijan Grand Prix,Baku City Circuit,2023
4,Miami Grand Prix,Miami International Autodrome,2023


Ejercicio 2: LEFT JOIN
Pregunta: ¿Cuáles son todos los pilotos y, si tienen resultados de carrera en la temporada 2021, muestra su posición final y los puntos obtenidos? Si un piloto no participó o no tiene resultados en 2021, debe aparecer en la lista con NULL en los campos de resultado.

Concepto: Un LEFT JOIN devuelve todas las filas de la tabla izquierda (drivers en este caso) y las filas coincidentes de la tabla derecha (results). Si no hay coincidencia en la tabla derecha, las columnas de la tabla derecha (positionText, points) contendrán NULL.

In [13]:
query = """ 
SELECT
    d.forename || ' ' || d.surname AS driver_name,
    ra.year AS race_year,
    ra.name AS race_name,
    res.positionText,
    res.points
FROM
    drivers AS d
LEFT JOIN
    results AS res ON d.driverId = res.driverId
LEFT JOIN
    races AS ra ON res.raceId = ra.raceId AND ra.year = 2021
WHERE
    (ra.year = 2021 OR ra.year IS NULL) -- Incluye pilotos sin resultados en 2021 pero presentes en la tabla 'drivers'
ORDER BY
    d.surname, ra.year DESC, res.points DESC
--LIMIT 15; -- Limitar para una visualización manejable
""" 
df = pd.read_sql_query(query, engine)

In [14]:
df[df.race_year == 2021]

,driver_name,race_year,race_name,positionText,points
775,Fernando Alonso,2021.0,Qatar Grand Prix,3,15.0
776,Fernando Alonso,2021.0,Hungarian Grand Prix,4,12.0
777,Fernando Alonso,2021.0,Azerbaijan Grand Prix,6,8.0
778,Fernando Alonso,2021.0,Dutch Grand Prix,6,8.0
779,Fernando Alonso,2021.0,Russian Grand Prix,6,8.0
...,...,...,...,...,...
24269,Sebastian Vettel,2021.0,Russian Grand Prix,12,0.0
24270,Sebastian Vettel,2021.0,Turkish Grand Prix,18,0.0
24271,Sebastian Vettel,2021.0,São Paulo Grand Prix,11,0.0
24272,Sebastian Vettel,2021.0,Saudi Arabian Grand Prix,R,0.0


Ejercicio 3: RIGHT JOIN (Emulado en SQLite)
Pregunta: ¿Cuáles son todos los constructores y, si tienen resultados registrados en la tabla constructor_results para la temporada 2023, muestra los puntos obtenidos? Si un constructor no tiene resultados en 2023, debe aparecer en la lista con NULL en los campos de resultado.

Concepto: Un RIGHT JOIN devuelve todas las filas de la tabla derecha (constructors en este caso) y las filas coincidentes de la tabla izquierda (constructor_results). SQLite no tiene RIGHT JOIN nativo. La forma de emularlo es invertir las tablas y usar un LEFT JOIN.

In [ ]:

query = """SELECT
    c.name AS constructor_name,
    cr.points,
    cr.raceId -- Opcional, para ver en qué carrera obtuvo los puntos
FROM
    constructors AS c
LEFT JOIN
    constructor_results AS cr ON c.constructorId = cr.constructorId
LEFT JOIN
    races AS r ON cr.raceId = r.raceId AND r.year = 2023
WHERE
    (r.year = 2023 OR r.year IS NULL) -- Para incluir constructores sin resultados en 2023
ORDER BY
    c.name ASC, cr.points DESC
LIMIT 15;"""

Ejercicio 4: FULL OUTER JOIN (Emulado en SQLite)
Pregunta: ¿Cuáles son todos los pilotos y todos los constructores, mostrando sus nombres, y si han participado en alguna carrera que tenga resultados registrados para ese año (ej. 2023) en la tabla results? Queremos ver pilotos sin constructores asociados a resultados en 2023 y constructores sin pilotos asociados a resultados en 2023.

Concepto: Un FULL OUTER JOIN devuelve todas las filas cuando hay una coincidencia en una de las tablas. Incluye filas no coincidentes de ambas tablas con NULL en las columnas no coincidentes. SQLite no soporta FULL OUTER JOIN directamente. Se emula usando un LEFT JOIN y un RIGHT JOIN (que a su vez es un LEFT JOIN invertido) combinados con UNION ALL.

In [ ]:
query = """-- Parte LEFT JOIN: Todos los pilotos y sus resultados de constructor en 2023
SELECT
    d.forename || ' ' || d.surname AS participant_name,
    'Driver' AS participant_type,
    co.name AS related_constructor_name,
    'Constructor' AS related_type,
    r.year AS relevant_year
FROM
    drivers AS d
LEFT JOIN
    results AS res ON d.driverId = res.driverId
LEFT JOIN
    constructors AS co ON res.constructorId = co.constructorId
LEFT JOIN
    races AS r ON res.raceId = r.raceId AND r.year = 2023
WHERE
    (r.year = 2023 OR r.year IS NULL) -- Solo filas relevantes para 2023 o pilotos sin resultados
GROUP BY
    d.driverId, co.constructorId, r.year -- Agrupamos para evitar duplicados si un piloto corre para el mismo constructor varias veces
HAVING
    COUNT(r.raceId) > 0 OR r.year IS NULL -- Solo mostrar si hay carreras en 2023 o si el piloto no tuvo resultados ese año

UNION ALL

-- Parte RIGHT JOIN (emulado): Todos los constructores y los pilotos asociados a ellos en 2023
SELECT
    co.name AS participant_name,
    'Constructor' AS participant_type,
    d.forename || ' ' || d.surname AS related_driver_name,
    'Driver' AS related_type,
    r.year AS relevant_year
FROM
    constructors AS co
LEFT JOIN
    results AS res ON co.constructorId = res.constructorId
LEFT JOIN
    drivers AS d ON res.driverId = d.driverId
LEFT JOIN
    races AS r ON res.raceId = r.raceId AND r.year = 2023
WHERE
    (r.year = 2023 OR r.year IS NULL)
GROUP BY
    co.constructorId, d.driverId, r.year
HAVING
    COUNT(r.raceId) > 0 OR r.year IS NULL
    AND d.driverId IS NULL -- Solo mostrar los constructores que no fueron ya cubiertos por la parte del piloto
ORDER BY participant_type DESC, participant_name
LIMIT 20; -- Limitar la salida, ya que FULL JOIN puede ser muy grande
"""

In [ ]:
docker desktop